# Pipeline 159 — Git-backed 10-run notebook

This notebook exposes the native evaluator loop. The active job is separate: open `Docker_Pipeline_10_Runs.live.log` to monitor it. Do not run these cells to attach to that job.

Runner/model checkpoints and experiment logs use Git; evaluator checkpoints use shared staging. Completed work is verified and skipped. No overwrite or local backup fallback.

In [ ]:
import fcntl
from urllib.request import Request, urlopen
import contextlib
import json
import os
from pathlib import Path
import pickle
import sys
import tempfile

sys.path.insert(0, '/helpers')
from tae_background_status import BackgroundWorkflowStatus
from p159_staged_continuation import (
    SharedStagingArchive, NotebookLog, SharedCohortStore, WorkflowStore,
    GoogleDriveBackupManager, QubitAllocator, AllocatorRunner,
    ExperimentConfiguration, MultiRunEvaluator,
)


## Verify the registered workflow and pinned checkpoints

This step reads existing artifacts; it does not start experiments.

In [ ]:
receipt = json.loads(Path('/run/tae/workflow-launch.json').read_bytes())
if (receipt.get('pipeline'), receipt.get('repository'), receipt.get('schema')) != (
    'Pipeline_159', 'pzg8794/TAE_Workflow', 'tae-docker-launch/v1'
):
    raise RuntimeError('Pipeline 159 launch receipt required')
execution = '3a1ba82a-1450-4e17-816d-917568e19736'
store = SharedCohortStore()
pinned = store.open_published('pipeline-159', execution, 'S1.5T')
key = 'Default/none/8'
finished = pinned.experiments[key]
completed_runner = pickle.loads(pinned.read(key, 'state'))
assert completed_runner['id'] == 8 and completed_runner['is_complete']
assert completed_runner['frames_count'] == 18000 and pinned.read(key, 'log')
for reference in finished.get('models', {}).values():
    store.read(pinned.head, reference['path'], reference['size'], reference['sha256'])
_, workflow_index, workflow = WorkflowStore().load(receipt['pipeline'])
if workflow_index.get(receipt['pipeline'], {}).get('version') != receipt['version']:
    raise RuntimeError('Launch receipt does not match the registered workflow version')
settings = workflow['settings']
models = settings['models']
assert set(completed_runner['results']) == set(models)
assert settings['runs'] == 10 and settings['base_frames'] == 4000 and settings['frame_step'] == 2000
binding = dict(pipelineId='159', driveId='0AE6N-7UYJfURUk9PVA')
drive = GoogleDriveBackupManager.pipeline_creation_drive_service()
evaluator_path = finished['evaluator']
observed = drive.files().get(fileId=evaluator_path['fileId'], supportsAllDrives=True,
    fields='id,name,driveId,parents,size,sha256Checksum,trashed').execute()
assert observed == dict(id=evaluator_path['fileId'], name=evaluator_path['name'],
    driveId=binding['driveId'], parents=[evaluator_path['parentId']], size=str(evaluator_path['size']),
    sha256Checksum=evaluator_path['sha256'], trashed=False)
print('Experiment 8: runner and log verified in pinned Git; evaluator verified on shared staging.', flush=True)
dashboard_log = Path('/tmp/pipeline-notebooks/Pipeline_159_Git_Resume.live.log')
missing = []
scenarios = {name: name for name in settings['threats']}
os.environ.update(DAQR_MODEL_MAX_WORKERS='4', DAQR_RESUME_ALLOW_DRIVE_DOWNLOADS='0')
native_stdout = sys.stdout


## Resume the native experiment loop

Run only when the previous runner has stopped and this is the registered container. Output is streamed to the cell and its live log. Completed positions without a matching checkpoint are reported, not rerun.

In [ ]:
def ensure_no_other_busy_kernel():
    from ipykernel import get_connection_file
    own_kernel = Path(get_connection_file()).stem.removeprefix('kernel-')
    servers = list((Path.home() / '.local/share/jupyter/runtime').glob('jpserver-*.json'))
    if not servers:
        raise RuntimeError('Cannot verify active Jupyter sessions')
    for server_file in servers:
        server = json.loads(server_file.read_bytes())
        request = Request(f"http://127.0.0.1:{int(server['port'])}/api/sessions",
                          headers={'Authorization': 'token ' + server['token']})
        with urlopen(request, timeout=5) as response:
            sessions = json.load(response)
        if any(session['kernel']['id'] != own_kernel and
               session['kernel']['execution_state'] == 'busy' for session in sessions):
            raise RuntimeError('Another notebook kernel is busy; no experiments started')

ensure_no_other_busy_kernel()
with open('/tmp/pipeline-159-notebook.lock', 'a') as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    with BackgroundWorkflowStatus(WorkflowStore(), receipt['pipeline'], execution) as reporter, dashboard_log.open('a', encoding='utf-8') as live:
        root = Path(tempfile.mkdtemp(prefix='pipeline-159-staging-'))
        for allocator_name in settings['allocators']:
            for base_cap in (False, True):
                for scale in settings['scales']:
                    cohort = f"S{scale:g}{'Tb' if base_cap else 'T'}"
                    if (allocator_name, cohort) == ('Default', 'S1T'):
                        print('Skipping verified completed Default S1T (50 positions).', file=NotebookLog(native_stdout, live))
                        continue
                    reporter.start(allocator_name, cohort)
                    run = store.open_run('pipeline-159', execution, cohort)
                    allocator = QubitAllocator() if allocator_name == 'Default' else AllocatorRunner(
                        allocator_name, ['default'], {}, [scale], [10], models, scenarios).create_allocator('default')
                    config = ExperimentConfiguration(runs=10, allocator=allocator, scenarios=scenarios,
                        models=models, scale=scale, base_capacity=base_cap, attack_intensity=settings['attack_intensity'],
                        suffix=cohort, resume=False, use_last_backup=False, overwrite=False, pipeline_storage=binding,
                        staging_dir=root, drive_service=drive)
                    config.resume_allow_drive_downloads = False
                    config.remote_experiment_archive = True
                    evaluator = MultiRunEvaluator(configs=config, base_frames=4000, frame_step=2000,
                                                  base_seed=settings['base_seed'])
                    archive = SharedStagingArchive(config, run, root)
                    evaluator._experiment_archive = archive
                    print(f'DOCKER PIPELINE: {allocator_name} {cohort} positions=10 frames=4000+2000',
                          file=NotebookLog(native_stdout, live))
                    saved_cohort = (workflow.get('allocators', {}).get(allocator_name, {})
                        .get('cohorts', {}).get(cohort, {}))
                    for threat in scenarios:
                        config.attack_type = threat
                        evaluator.env_experiments.setdefault(threat, {})
                        for position in range(1, 11):
                            key = f'{allocator_name}/{threat}/{position}'
                            prior = run.experiments.get(key)
                            if prior is not None:
                                state = pickle.loads(run.read(key, 'state'))
                                assert state['id'] == position and state['is_complete'] and set(state['results']) == set(models)
                                evaluator.env_experiments[threat][position] = dict(
                                    results=state['results'], winner=state['winner'], exp_id=position)
                                print(f'SKIPPING EXPERIMENT {position}: ALREADY COMPLETED AND STORED',
                                      file=NotebookLog(native_stdout, live))
                                continue
                            status = saved_cohort.get('threats', {}).get(threat, {}).get('experiments', {}).get(str(position), {})
                            if status.get('state') == 'done':
                                missing.append(key)
                                print(f'{key}: skipped completed experiment with missing checkpoint',
                                      file=NotebookLog(native_stdout, live))
                                continue
                            with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', dir=root, prefix='experiment-log-') as experiment_log:
                                archive.log_path, archive.log_stream = Path(experiment_log.name), experiment_log
                                with contextlib.redirect_stdout(NotebookLog(NotebookLog(native_stdout, live), experiment_log)):
                                    evaluator.run_experiment(exp_no=position-1)
                                    print(f'{key}: checkpoint, log, and evaluator verified remotely.')
                    if all(set(evaluator.env_experiments[name]) == set(range(1, 11)) for name in scenarios):
                        evaluator.calculate_scenarios_performance()
                        reporter.finish_cohort(complete=True)
                    else:
                        reporter.finish_cohort(complete=False)
                        print(f'{allocator_name}/{cohort}: incomplete evidence; evaluation not marked complete.',
                              file=NotebookLog(native_stdout, live))
    if missing:
        print('Completed positions skipped with missing Git artifacts:', missing)
